# Nível 1 — Dados e primeira análise com LLM

Este notebook implementa o tratamento dos dados, a normalização dos valores para BRL, as regras determinísticas de sinalização e a análise de um caso selecionado com LLM.



## Carga dos dados

Nesta etapa, carregamos o arquivo `dados_nivel_1.json`, a taxa de câmbio fornecida e as operações em um DataFrame pandas.

In [5]:
import pandas as pd
import json

with open('../dados/dados_nivel_1.json', 'r', encoding='utf-8') as f:
    dados = json.load(f)

taxa_cambio = dados['taxa_cambio_usd_brl']
df = pd.DataFrame(dados['operacoes'])

print(f"Taxa de câmbio USD/BRL: {taxa_cambio}")
print(f"Total de operações carregadas: {len(df)}")
df.head(10)

Taxa de câmbio USD/BRL: 5.4
Total de operações carregadas: 20


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,
5,OP-0006,CLI-A-2,2026-03-14,27000,BRL,ted,transferencia_enviada,Delta Transportes,
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
7,OP-0008,CLI-A-3,2026-03-05,15200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
8,OP-0009,CLI-A-3,2026-03-05,16100,BRL,pix,transferencia_enviada,Zeta Importacao,
9,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,


## Limpeza e tratamento dos dados

Antes de qualquer agregação ou regra, vamos investigar a qualidade dos dados brutos. 
Vamos checar: registros duplicados, valores em moeda diferente de BRL, e datas ausentes.

In [6]:
import numpy as np

# Cria uma cópia para preservar o dataframe original carregado
df_clean = df.copy()

# 1. Tratamento: Remoção de duplicatas exatas pelo 'id'
df_clean = df_clean.drop_duplicates(subset=['id'], keep='first')

# 2. Tratamento: Normalização de valores para BRL
df_clean['valor_brl'] = np.where(
    df_clean['moeda'] == 'USD',
    df_clean['valor'] * taxa_cambio,
    df_clean['valor']
)

# 3. Tratamento: Conversão de datas (valores nulos viram NaT nativamente)
df_clean['data'] = pd.to_datetime(df_clean['data'])

print(f"Total de operações após limpeza: {len(df_clean)}")

# Mostrando especificamente os casos tratados para validar a limpeza
casos_tratados = ['OP-0007', 'OP-0013', 'OP-0017']
df_clean[df_clean['id'].isin(casos_tratados)][['id', 'cliente_id', 'data', 'valor', 'moeda', 'valor_brl']]

Total de operações após limpeza: 19


,id,cliente_id,data,valor,moeda,valor_brl
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,17200.0
13,OP-0013,CLI-A-4,2026-03-24,12000,USD,64800.0
17,OP-0017,CLI-A-5,NaT,4300,BRL,4300.0


### Justificativa das Decisões de Tratamento

Conforme os problemas levantados, tomamos as seguintes decisões:

1. **Registros Duplicados (`OP-0007`):** Removemos a duplicata mantendo apenas a primeira ocorrência. Um sistema transacional não deve ter o mesmo ID de operação para transações distintas.
2. **Moeda Estrangeira (`OP-0013` em USD):** Criamos a coluna `valor_brl`. Multiplicamos o valor original pela taxa de câmbio fixa (5.4) fornecida no arquivo, normalizando a base para as regras futuras.
3. **Data Faltante (`OP-0017` com data nula):** Ao converter para datetime, o valor se tornou `NaT`. Decidimos **não excluir** a linha. Isso garante que o volume financeiro total deste cliente não seja prejudicado na etapa de agregações. A operação apenas não será contabilizada em regras que dependam estritamente do dia (como a Regra 1).

## Agregações

Com os dados tratados e os valores normalizados para BRL, calculamos o volume total transacionado por cliente e a quantidade de operações por canal.

In [7]:
volume_por_cliente = (
    df_clean.groupby("cliente_id")["valor_brl"]
    .sum()
    .sort_values(ascending=False)
)

print("Volume total transacionado por cliente:")
display(volume_por_cliente)

operacoes_por_canal = (
    df_clean.groupby("canal")
    .size()
    .sort_values(ascending=False)
)

print("Quantidade de operações por canal:")
display(operacoes_por_canal)

Volume total transacionado por cliente:


cliente_id
CLI-A-4    79500.0
CLI-A-1    57500.0
CLI-A-2    52900.0
CLI-A-3    48500.0
CLI-A-5    16900.0
CLI-A-6    10200.0
Name: valor_brl, dtype: float64

Quantidade de operações por canal:


canal
pix        8
ted        5
boleto     3
cartao     2
especie    1
dtype: int64

## Regras determinísticas

Implementamos as duas regras de sinalização exigidas, adicionando as flags ao DataFrame.
Todo o cálculo (soma, mediana, comparação com limite) é feito em pandas, a LLM não participa dessa etapa.

In [8]:
# ===== Regra 1 — Fracionamento =====
# Cliente com 3+ operações na mesma data, soma > R$50.000,
# e nenhuma operação isolada >= R$20.000

agrupado = df_clean.groupby(['cliente_id', 'data'])['valor_brl'].agg(
    qtd_operacoes='count',
    soma_valores='sum',
    maior_operacao='max'
).reset_index()

agrupado['flag_fracionamento'] = (
    (agrupado['qtd_operacoes'] >= 3) &
    (agrupado['soma_valores'] > 50000) &
    (agrupado['maior_operacao'] < 20000)
)

clientes_fracionamento = agrupado[agrupado['flag_fracionamento']]['cliente_id'].unique()
print("Clientes sinalizados na Regra 1 (Fracionamento):")
print(clientes_fracionamento)
print()

# Adiciona a flag ao df_clean (nível cliente+data)
df_clean = df_clean.merge(
    agrupado[['cliente_id', 'data', 'flag_fracionamento']],
    on=['cliente_id', 'data'],
    how='left'
)

# ===== Regra 2 — Valor atípico =====
# Operação > 5x a mediana do cliente, só para clientes com 4+ operações

contagem_cliente = df_clean.groupby('cliente_id')['valor_brl'].transform('count')
mediana_cliente = df_clean.groupby('cliente_id')['valor_brl'].transform('median')

df_clean['flag_valor_atipico'] = (
    (contagem_cliente >= 4) &
    (df_clean['valor_brl'] > 5 * mediana_cliente)
)

print("Operações sinalizadas na Regra 2 (Valor atípico):")
print(df_clean[df_clean['flag_valor_atipico']][['id', 'cliente_id', 'valor_brl']])

Clientes sinalizados na Regra 1 (Fracionamento):
<StringArray>
['CLI-A-1']
Length: 1, dtype: str

Operações sinalizadas na Regra 2 (Valor atípico):
         id cliente_id  valor_brl
12  OP-0013    CLI-A-4    64800.0


### Validação da Regra 1 (Fracionamento)

Comparamos dois casos parecidos para mostrar que a regra funciona corretamente:
CLI-A-1 (deveria ser sinalizado) e CLI-A-3 (não deveria, mesmo tendo padrão similar).

In [9]:
casos_validacao = agrupado[
    (agrupado['cliente_id'].isin(['CLI-A-1', 'CLI-A-3'])) &
    (agrupado['qtd_operacoes'] >= 3)
]

print(casos_validacao[
    ['cliente_id', 'data', 'qtd_operacoes', 'soma_valores',
     'maior_operacao', 'flag_fracionamento']
])

  cliente_id       data  qtd_operacoes  soma_valores  maior_operacao  \
0    CLI-A-1 2026-03-09              3       54200.0         18800.0   
3    CLI-A-3 2026-03-05              3       48500.0         17200.0   

   flag_fracionamento  
0                True  
3               False  


## Parte B — Análise com LLM

Nesta etapa, utilizamos uma LLM para interpretar um cliente previamente sinalizado pelas regras determinísticas.

Os cálculos e a identificação das sinalizações permanecem sob responsabilidade do pandas. A LLM recebe os resultados já calculados e é utilizada apenas para interpretação e elaboração do parecer.

Para a análise, foi selecionado o cliente `CLI-A-1`, sinalizado pela regra de fracionamento.

In [10]:
import os
import time
from dotenv import load_dotenv
from google import genai

load_dotenv("../.env")

api_key = os.getenv("GOOGLE_API_KEY")

if not api_key:
    raise ValueError("GOOGLE_API_KEY não encontrada no arquivo .env")

client = genai.Client(api_key=api_key)

MODELO = "gemini-3.6-flash"

print(f"Cliente Gemini configurado com sucesso.")
print(f"Modelo selecionado: {MODELO}")

Cliente Gemini configurado com sucesso.
Modelo selecionado: gemini-3.6-flash


### Seleção e preparação do cliente

Para a análise com LLM, foi selecionado o cliente `CLI-A-1`, previamente sinalizado pela Regra 1 (Fracionamento).

A seguir, extraímos seu histórico de operações já tratado e normalizado para BRL. Os cálculos e a sinalização foram realizados previamente com pandas; a LLM receberá essas informações como contexto para interpretação do caso.

In [11]:
cliente_selecionado = "CLI-A-1"

historico_cliente = df_clean[
    df_clean["cliente_id"] == cliente_selecionado
][
    [
        "id",
        "data",
        "valor_brl",
        "canal",
        "tipo",
        "contraparte",
        "observacao"
    ]
].copy()

display(historico_cliente)

,id,data,valor_brl,canal,tipo,contraparte,observacao
0,OP-0001,2026-03-09,18100.0,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,2026-03-09,17300.0,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,2026-03-09,18800.0,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,2026-03-21,3300.0,boleto,pagamento,Gama Distribuidora,


### Estrutura do parecer

A resposta da LLM será solicitada em formato estruturado, com os campos exigidos pelo desafio: `nivel_risco`, `tipologia_suspeita`, `red_flags` e `justificativa`.

Essa estrutura facilita a validação da resposta e permite tratar casos em que o retorno da LLM esteja incompleto ou malformado.


In [13]:
from pydantic import BaseModel
from typing import Literal

class ParecerPLD(BaseModel):
    nivel_risco: Literal["baixo", "médio", "alto"]
    tipologia_suspeita: str
    red_flags: list[str]
    justificativa: str

### Prompt — Versão 1

A primeira versão do prompt fornece à LLM o histórico do cliente e os resultados já calculados pela regra determinística. O modelo é instruído a interpretar os fatos fornecidos, sem refazer cálculos ou substituir a sinalização realizada em pandas.

In [14]:
caso_fracionamento = agrupado[
    (agrupado["cliente_id"] == cliente_selecionado) &
    (agrupado["flag_fracionamento"])
].iloc[0]

contexto_operacoes = historico_cliente.to_string(index=False)

prompt_v1 = f"""
Você atua como analista de triagem de Prevenção à Lavagem de Dinheiro (PLD).

Analise o caso abaixo e produza um parecer de triagem.
Não refaça cálculos e não determine novamente se a regra foi atendida.
Os resultados numéricos abaixo já foram calculados de forma determinística
com pandas e devem ser tratados como fatos.

Cliente: {cliente_selecionado}

Resultado da regra determinística:
- Regra acionada: Fracionamento
- Quantidade de operações no dia: {int(caso_fracionamento["qtd_operacoes"])}
- Soma das operações no dia: R$ {caso_fracionamento["soma_valores"]:.2f}
- Maior operação individual: R$ {caso_fracionamento["maior_operacao"]:.2f}
- Resultado: cliente sinalizado

Histórico de operações:
{contexto_operacoes}

Com base exclusivamente nas informações fornecidas, avalie o nível de risco
e explique os elementos relevantes para uma eventual análise humana.
Não trate a sinalização como prova de lavagem de dinheiro.
"""

print(prompt_v1)


Você atua como analista de triagem de Prevenção à Lavagem de Dinheiro (PLD).

Analise o caso abaixo e produza um parecer de triagem.
Não refaça cálculos e não determine novamente se a regra foi atendida.
Os resultados numéricos abaixo já foram calculados de forma determinística
com pandas e devem ser tratados como fatos.

Cliente: CLI-A-1

Resultado da regra determinística:
- Regra acionada: Fracionamento
- Quantidade de operações no dia: 3
- Soma das operações no dia: R$ 54200.00
- Maior operação individual: R$ 18800.00
- Resultado: cliente sinalizado

Histórico de operações:
     id       data  valor_brl  canal                  tipo        contraparte observacao
OP-0001 2026-03-09    18100.0    pix transferencia_enviada Alfa Comercio LTDA           
OP-0002 2026-03-09    17300.0    pix transferencia_enviada Alfa Comercio LTDA           
OP-0003 2026-03-09    18800.0    ted transferencia_enviada   Beta Servicos ME           
OP-0004 2026-03-21     3300.0 boleto             pagamento 

### Execução do Prompt — Versão 1

O prompt é enviado ao modelo utilizando o schema `ParecerPLD` como formato esperado de resposta. A execução registra a latência e o consumo de tokens.

A resposta retornada é validada com Pydantic. Caso o conteúdo não possa ser validado conforme o schema definido, o erro é tratado explicitamente.

In [15]:
from google.genai import types
from pydantic import ValidationError

inicio = time.perf_counter()

try:
    resposta_v1 = client.models.generate_content(
        model=MODELO,
        contents=prompt_v1,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=ParecerPLD,
        ),
    )

    tempo_v1 = time.perf_counter() - inicio

    parecer_v1 = ParecerPLD.model_validate_json(resposta_v1.text)

    print("Parecer validado com sucesso:")
    print(parecer_v1.model_dump_json(indent=2))

except (ValidationError, ValueError) as erro:
    tempo_v1 = time.perf_counter() - inicio
    parecer_v1 = None

    print("Resposta malformada ou incompatível com o schema.")
    print(f"Erro: {erro}")

except Exception as erro:
    tempo_v1 = time.perf_counter() - inicio
    parecer_v1 = None

    print("Erro durante a chamada à API.")
    print(f"Erro: {erro}")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Parecer validado com sucesso:
{
  "nivel_risco": "médio",
  "tipologia_suspeita": "Fracionamento ou Smurfing",
  "red_flags": [
    "Múltiplas transferências de alto valor em um único dia (09/03/2026)",
    "Valores fracionados de montante expressivo (entre R$ 17.300 e R$ 18.800)",
    "Uso de múltiplos canais de pagamento (PIX e TED) no mesmo dia para diferentes contrapartes"
  ],
  "justificativa": "O cliente realizou três transações enviadas em um único dia (09/03/2026) somando R$ 54.200,00, direcionadas a duas contrapartes distintas (Alfa Comercio LTDA e Beta Servicos ME) por meio de PIX e TED. O padrão de valores elevados repetidos em curto espaço de tempo se alinha à suspeita de fracionamento para evitar limites de controle, exigindo verificação humana sobre a compatibilidade patrimonial/faturamento e a finalidade econômica das transações."
}


### Métricas da execução — Versão 1

A seguir, registramos a latência e o consumo de tokens da chamada, utilizando os metadados retornados pela própria API.

In [17]:
print(f"Tempo de resposta: {tempo_v1:.2f} segundos")

if resposta_v1.usage_metadata:
    uso_v1 = resposta_v1.usage_metadata

    print(f"Tokens de entrada: {uso_v1.prompt_token_count}")
    print(f"Tokens de saída: {uso_v1.candidates_token_count}")
    print(f"Tokens de raciocínio: {uso_v1.thoughts_token_count}")
    print(f"Tokens totais: {uso_v1.total_token_count}")

Tempo de resposta: 6.18 segundos
Tokens de entrada: 380
Tokens de saída: 234
Tokens de raciocínio: 703
Tokens totais: 1317


### Prompt — Versão 2

A segunda versão torna as instruções mais restritivas quanto ao uso das evidências disponíveis. O modelo deve distinguir fatos observados de hipóteses, evitar inferir intenções do cliente e fundamentar as red flags exclusivamente nos dados fornecidos.

O objetivo é avaliar se instruções mais específicas produzem um parecer mais cauteloso e rastreável.

In [18]:
prompt_v2 = f"""
Você atua como analista de triagem de Prevenção à Lavagem de Dinheiro (PLD).

Produza um parecer de triagem sobre o cliente abaixo.

Regras para a análise:
1. Não refaça cálculos e não determine novamente se a regra foi atendida.
2. Considere os resultados determinísticos fornecidos como fatos já calculados pelo pandas.
3. Baseie cada red flag somente em informações presentes nos dados fornecidos.
4. Não presuma intenção, origem dos recursos, relacionamento entre as partes ou finalidade das operações quando essas informações não estiverem disponíveis.
5. Diferencie padrões observados de hipóteses que exigiriam investigação adicional.
6. A sinalização indica necessidade de análise humana e não constitui prova de lavagem de dinheiro.
7. Na justificativa, indique também quais informações adicionais seriam úteis para aprofundar a análise.

Cliente: {cliente_selecionado}

Resultado da regra determinística:
- Regra acionada: Fracionamento
- Quantidade de operações no dia: {int(caso_fracionamento["qtd_operacoes"])}
- Soma das operações no dia: R$ {caso_fracionamento["soma_valores"]:.2f}
- Maior operação individual: R$ {caso_fracionamento["maior_operacao"]:.2f}
- Resultado: cliente sinalizado

Histórico de operações:
{contexto_operacoes}

Produza o parecer exclusivamente a partir dessas informações.
"""

print(prompt_v2)


Você atua como analista de triagem de Prevenção à Lavagem de Dinheiro (PLD).

Produza um parecer de triagem sobre o cliente abaixo.

Regras para a análise:
1. Não refaça cálculos e não determine novamente se a regra foi atendida.
2. Considere os resultados determinísticos fornecidos como fatos já calculados pelo pandas.
3. Baseie cada red flag somente em informações presentes nos dados fornecidos.
4. Não presuma intenção, origem dos recursos, relacionamento entre as partes ou finalidade das operações quando essas informações não estiverem disponíveis.
5. Diferencie padrões observados de hipóteses que exigiriam investigação adicional.
6. A sinalização indica necessidade de análise humana e não constitui prova de lavagem de dinheiro.
7. Na justificativa, indique também quais informações adicionais seriam úteis para aprofundar a análise.

Cliente: CLI-A-1

Resultado da regra determinística:
- Regra acionada: Fracionamento
- Quantidade de operações no dia: 3
- Soma das operações no dia: R

In [ ]:
cliente_selecionado = "CLI-A-1"

historico_cliente = df_clean[
    df_clean["cliente_id"] == cliente_selecionado
][
    [
        "id",
        "data",
        "valor_brl",
        "canal",
        "tipo",
        "contraparte",
        "observacao"
    ]
].copy()

display(historico_cliente)

,id,data,valor_brl,canal,tipo,contraparte,observacao
0,OP-0001,2026-03-09,18100.0,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,2026-03-09,17300.0,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,2026-03-09,18800.0,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,2026-03-21,3300.0,boleto,pagamento,Gama Distribuidora,


### Execução do Prompt — Versão 2

A segunda versão do prompt é executada utilizando o mesmo schema de resposta da versão anterior. Novamente, são registrados a latência, o consumo de tokens e a validação do parecer retornado.

In [19]:
inicio = time.perf_counter()

try:
    resposta_v2 = client.models.generate_content(
        model=MODELO,
        contents=prompt_v2,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=ParecerPLD,
        ),
    )

    tempo_v2 = time.perf_counter() - inicio

    parecer_v2 = ParecerPLD.model_validate_json(resposta_v2.text)

    print("Parecer validado com sucesso:")
    print(parecer_v2.model_dump_json(indent=2))

except (ValidationError, ValueError) as erro:
    tempo_v2 = time.perf_counter() - inicio
    parecer_v2 = None

    print("Resposta malformada ou incompatível com o schema.")
    print(f"Erro: {erro}")

except Exception as erro:
    tempo_v2 = time.perf_counter() - inicio
    parecer_v2 = None

    print("Erro durante a chamada à API.")
    print(f"Erro: {erro}")

Parecer validado com sucesso:
{
  "nivel_risco": "médio",
  "tipologia_suspeita": "Fracionamento de Operações (Structuring)",
  "red_flags": [
    "Concentração de 3 transferências enviadas em um único dia (09/03/2026) somando R$ 54.200,00",
    "Múltiplas transferências via PIX para a mesma contraparte (Alfa Comercio LTDA) no mesmo dia",
    "Operações com valores individuais próximos e elevados (R$ 17.300,00, R$ 18.100,00 e R$ 18.800,00)"
  ],
  "justificativa": "O cliente foi sinalizado pelo acionamento da regra determinística de fracionamento após realizar, no dia 09/03/2026, três movimentações de saída no total de R$ 54.200,00, sendo duas via PIX para Alfa Comercio LTDA (R$ 18.100,00 e R$ 17.300,00) e uma via TED para Beta Servicos ME (R$ 18.800,00). O padrão de fragmentação de valores elevados em um mesmo dia sugere uma hipótese de atipicidade a ser investigada, destacando-se que a sinalização é um alerta para análise humana e não comprova prática de lavagem de dinheiro. Para apr

### Métricas da execução — Versão 2

Registramos novamente a latência e o consumo de tokens para permitir a comparação com a primeira versão do prompt.


In [20]:
print(f"Tempo de resposta: {tempo_v2:.2f} segundos")

if resposta_v2.usage_metadata:
    uso_v2 = resposta_v2.usage_metadata

    print(f"Tokens de entrada: {uso_v2.prompt_token_count}")
    print(f"Tokens de saída: {uso_v2.candidates_token_count}")
    print(f"Tokens de raciocínio: {uso_v2.thoughts_token_count}")
    print(f"Tokens totais: {uso_v2.total_token_count}")

Tempo de resposta: 34.53 segundos
Tokens de entrada: 463
Tokens de saída: 380
Tokens de raciocínio: 1801
Tokens totais: 2644


### Comparação entre os prompts

As duas versões classificaram o cliente como risco médio e identificaram o padrão de fracionamento como principal tipologia suspeita.

A principal diferença ocorreu na forma de justificar o parecer. Na Versão 1, o modelo associou o padrão à intenção de "evitar limites de controle", embora essa intenção não possa ser comprovada com os dados disponíveis.

Na Versão 2, as instruções mais restritivas produziram uma resposta mais cautelosa e fundamentada nas evidências observadas. O modelo tratou o comportamento como uma hipótese a ser investigada, deixou explícito que a sinalização não comprova lavagem de dinheiro e indicou informações adicionais úteis para aprofundar a análise, como perfil socioeconômico, atividade do cliente, relação com as contrapartes e documentos de suporte.

Em contrapartida, a Versão 2 apresentou maior custo computacional: o total de tokens aumentou de 1.317 para 2.644, e a latência passou de 6,18 para 34,53 segundos.

Neste caso, a Versão 2 produziu um parecer mais adequado para triagem, porém com maior custo e latência. Em um cenário de produção, esse trade-off deveria ser considerado de acordo com o volume de casos e o nível de rigor necessário.